# [Hands-On] Understanding Reinforcement Learning and Implementation (Text Version)

- Author : Sangkeun Jung (hugmanskj@gmail.com)

> Educational Purpose

This tutorial demonstrates how to train a Deep Q-Network (DQN) on textual representations of the CartPole environment.

Key points include:

1. Converting numeric CartPole states into text form via a custom Gym wrapper
2. Encoding those text observations with DistilBERT
3. Implementing a replay buffer and DQN training loop
4. Visualizing intermediate states as images (rendered environment) and textual info, then compiling them into a GIF

# Environment Setup
Install the required packages (if not already installed):

In [1]:
!pip install gymnasium numpy torch torchvision transformers pillow imageio

# Library Imports

In [2]:
import gymnasium as gym
import os
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import imageio

from transformers import DistilBertTokenizer, DistilBertModel
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPyImage
%matplotlib inline

ModuleNotFoundError: No module named 'gymnasium'

In [6]:
!pip install gym

In [3]:
import sys

In [4]:
sys.executable

'C:\\Users\\hugman\\anaconda3\\python.exe'

In [6]:
sys.path

['C:\\Users\\hugman\\anaconda3\\python312.zip',
 'C:\\Users\\hugman\\anaconda3\\DLLs',
 'C:\\Users\\hugman\\anaconda3\\Lib',
 'C:\\Users\\hugman\\anaconda3',
 '',
 'C:\\Users\\hugman\\anaconda3\\Lib\\site-packages',
 'C:\\Users\\hugman\\anaconda3\\Lib\\site-packages\\win32',
 'C:\\Users\\hugman\\anaconda3\\Lib\\site-packages\\win32\\lib',
 'C:\\Users\\hugman\\anaconda3\\Lib\\site-packages\\Pythonwin',
 'C:\\Users\\hugman\\anaconda3\\Lib\\site-packages\\setuptools\\_vendor']

- gymnasium: For the CartPole environment
- torch / transformers: Neural network and DistilBERT implementation
- imageio: For saving sequences of images/GIFs
- PIL (Pillow): Image processing
- matplotlib: Optional for plotting

# State Management
## StateStorage Class
One key innovation in our implementation is stacking multiple states to provide temporal context to our agent. Here's how we manage state history:

In [ ]:
class StateStorage:
   """
   A class that stores n most recent states and returns them as a stacked tensor.
   Example: if n=4, always provides states in form [s(t-3), s(t-2), s(t-1), s(t)]
   """
   def __init__(self, state_dim, stack_size=4):
       """
       Args:
           state_dim (int): Dimension of a single state (4 for CartPole)
           stack_size (int): Number of recent states to stack
       """
       self.stack_size = stack_size
       self.state_dim = state_dim

       # Deque to store states
       # Maximum length = stack_size
       self.storage = deque(maxlen=stack_size)

       # Pre-fill with zero vectors for initialization
       for _ in range(stack_size):
           self.storage.append(np.zeros(state_dim, dtype=np.float32))

   def push(self, state):
       """
       Store a new state (single). The oldest state is automatically removed (deque)
       """
       self.storage.append(state)

   def get_stacked_state(self):
       """
       Combines all currently stacked states and returns in shape (stack_size * state_dim,)
       (or 2D shape: [stack_size, state_dim])
       """
       # Example: stack_size=4, state_dim=4 -> shape=(4,4)
       stacked = np.array(self.storage, dtype=np.float32)
       # Here we flatten to (stack_size * state_dim)
       stacked = stacked.reshape(-1)  # shape: (4*4,) = (16,)
       return stacked

- Frame stacking for images would increase input channels (e.g., stacking 4 frames).
- Not mandatory for simple tasks but often beneficial in more complex environments.

# Neural Network Architecture

## TextCartPoleWrapper: Convert Numeric Observations to Text

This Gym wrapper transforms the 4D numeric state of CartPole into a textual description,
- e.g., obs = [-0.02, 0.05, 0.10, -0.04] → "Cart Position=-0.02, Velocity=0.05, Pole Angle=5.73 deg, Angular Vel=-0.04"

In [ ]:
class TextCartPoleWrapper(gym.Wrapper):
   """
   A wrapper for CartPole that converts observation values (4D vector) into text sentences.
   Example: obs = [-0.02, 0.05, 0.10, -0.04]
           -> "Cart position=-0.02, velocity=0.05, pole angle=5.73deg, angular velocity=-0.04"
   """
   def __init__(self, env):
       super().__init__(env)

   def numeric_to_text(self, obs):
       # obs: [cart_position, cart_velocity, pole_angle, pole_angular_velocity]
       cart_pos, cart_vel, pole_angle, pole_ang_vel = obs

       # Convert pole_angle from rad to deg
       deg = pole_angle * (180.0 / np.pi)
       text_obs = (
           f"Cart position={cart_pos:.3f}, velocity={cart_vel:.3f}, "
           f"pole angle={deg:.2f}deg, angular velocity={pole_ang_vel:.3f}"
       )
       return text_obs

   def reset(self, **kwargs):
       obs, info = self.env.reset(**kwargs)
       text_obs = self.numeric_to_text(obs)
       return text_obs, info

   def step(self, action):
       obs, reward, done, truncated, info = self.env.step(action)
       text_obs = self.numeric_to_text(obs)
       return text_obs, reward, done, truncated, info

#### Result
Calling env.reset() or env.step() now returns a string instead of a numeric array.

## DistilBERT-Based Text Encoder
We will use DistilBERT to embed each text observation into a vector.

In [ ]:
class DistilBertEncoder(nn.Module):
    def __init__(self, pretrained_model_name="distilbert-base-uncased", embed_dim=768, freeze_bert=True):
        super().__init__()
        self.tokenizer = DistilBertTokenizer.from_pretrained(pretrained_model_name)
        self.bert = DistilBertModel.from_pretrained(pretrained_model_name)

        # (Important) Freeze pre-trained BERT parameters
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        # Either reduce BERT's default output (768 dim) to desired embedding dimension or use as is
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, text_list):
        encoding = self.tokenizer(text_list, return_tensors="pt",
                                padding=True, truncation=True, max_length=64)
        input_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state
        cls_embedding = hidden_state[:, 0, :]  # (batch_size, 768)
        # Only fc layer is trained if BERT is frozen
        out = self.fc(cls_embedding)
        return out

- DistilBERT is a lightweight Transformer model suitable for shorter text.
- Here, we use the hidden state of the first token (index 0) as a sentence embedding.

# PolicyNetwork (DQN)

In [ ]:
class DQN(nn.Module):
   """
   DistilBertEncoder -> Q-head
   """
   def __init__(self, action_dim, bert_embed_dim=768):
       super().__init__()
       self.encoder = DistilBertEncoder(embed_dim=bert_embed_dim)
       self.q_head = nn.Linear(bert_embed_dim, action_dim)

   def forward(self, text_list):
       """
       text_list: List[str] (batch_size sentences)
       return: Q-values of shape (batch_size, action_dim)
       """
       emb = self.encoder(text_list)  # (batch_size, bert_embed_dim)
       q = self.q_head(emb)
       return q

- forward: Takes a batch of text strings, encodes them via DistilBertEncoder, and outputs Q-values.


# ReplayBuffer
- Since our state is now a string, we store (state_str, action, reward, next_state_str, done) in the buffer.
- When sampling, we produce a list of strings for both states and next_states.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state_str, action, reward, next_state_str, done):
        self.buffer.append((state_str, action, reward, next_state_str, done))

    def sample(self, batch_size):
        batch = np.random.choice(len(self.buffer), batch_size, replace=False)
        samples = [self.buffer[idx] for idx in batch]
        states, actions, rewards, next_states, dones = zip(*samples)
        return list(states), actions, rewards, list(next_states), dones

    def __len__(self):
        return len(self.buffer)


# DQN Training Loop

#### store_frames_and_info function
Similar to the previous image-based example, we can save rendered frames plus textual info for each step:

In [ ]:
def store_frames_and_info(episode_number, frames, info_texts, render_path):
   """
   Args:
       episode_number (int): Episode number
       frames (list of np.array): Array of rendered images
       info_texts (list of str): Information strings for each step
       render_path (str): Parent directory path for saving
   """
   ep_folder = os.path.join(render_path, f"episode_{episode_number}")
   os.makedirs(ep_folder, exist_ok=True)

   for i, (img, info_str) in enumerate(zip(frames, info_texts)):
       # step_xxx.png
       png_path = os.path.join(ep_folder, f"step_{i:03d}.png")
       imageio.imwrite(png_path, img)

       # step_xxx.info.txt
       info_path = os.path.join(ep_folder, f"step_{i:03d}.info.txt")
       with open(info_path, "w", encoding="utf-8") as f:
           f.write(info_str)


In [ ]:
def train_dqn(
    env_name="CartPole-v1",
    num_episodes=100,
    batch_size=32,
    gamma=0.99,
    lr=1e-3,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=500,
    target_update_interval=20,
    buffer_capacity=50000,
    save_render=True,
    render_path="./cartpole_text_frames"
):
    """
    DQN training function using text modality.
    Pipeline: CartPole -> TextCartPoleWrapper -> DistilBERT -> Q.
    """
    # 1) Create environment + text wrapper
    base_env = gym.make(env_name, render_mode="rgb_array")
    env = TextCartPoleWrapper(base_env)

    # Observation is string (e.g., "Cart position=..., Pole angle=..."), action dim is 2
    action_dim = env.action_space.n

    # 2) Models & buffer
    dqn = DQN(action_dim, bert_embed_dim=768)
    target_dqn = DQN(action_dim, bert_embed_dim=768)
    target_dqn.load_state_dict(dqn.state_dict())

    optimizer = optim.Adam(dqn.parameters(), lr=lr)
    replay_buffer = ReplayBuffer(capacity=buffer_capacity)

    # 3) Epsilon setup
    epsilon = epsilon_start
    epsilon_decay_rate = (epsilon_start - epsilon_end) / epsilon_decay

    # 4) Folder setup
    if save_render and not os.path.exists(render_path):
        os.makedirs(render_path)

    total_rewards_list = []

    for episode in range(num_episodes):
        # reset -> text state
        state_str, _ = env.reset()
        done = False
        ep_reward = 0

        # For storing frames & info.txt
        episode_frames = []
        episode_info_texts = []
        episode_state_texts = []
        step_count = 0

        while not done:
            # \(\epsilon\)-greedy
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                # DistilBERT needs batch input → [state_str] (single item)
                state_batch = [state_str]
                with torch.no_grad():
                    q_values = dqn(state_batch)  # shape: (1, action_dim)
                    action = q_values.argmax(dim=1).item()

            # step
            next_state_str, reward, done, truncated, info = env.step(action)
            done = done or truncated
            ep_reward += reward

            replay_buffer.push(state_str, action, reward, next_state_str, done)

            # Collect rendering & info
            if save_render:
                frame = env.render()  # (H, W, 3)
                episode_frames.append(frame)

                info_str = (
                    f"Episode={episode+1}, Step={step_count}\n"
                    f"Action={action}, Reward={reward:.2f}\n"
                )
                episode_state_texts.append(f"State Text=\"{state_str}\"\n")
                episode_info_texts.append(info_str)

            state_str = next_state_str
            step_count += 1

            # Training
            if len(replay_buffer) > batch_size:
                states_list, actions, rewards, next_states_list, dones = replay_buffer.sample(batch_size)

                actions_t = torch.LongTensor(actions).unsqueeze(1)  # (batch, 1)
                rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
                dones_t = torch.BoolTensor(dones).unsqueeze(1)

                # Q(s, a)
                q_pred = dqn(states_list).gather(1, actions_t)

                with torch.no_grad():
                    q_next = target_dqn(next_states_list).max(dim=1, keepdim=True)[0]
                    q_target = rewards_t + gamma * q_next * (~dones_t)

                loss = F.mse_loss(q_pred, q_target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                # Decrease epsilon
                if epsilon > epsilon_end:
                    epsilon -= epsilon_decay_rate

        # Episode end
        total_rewards_list.append(ep_reward)

        # Update target network
        if (episode+1) % target_update_interval == 0:
            target_dqn.load_state_dict(dqn.state_dict())

        # Log/save every 10 episodes
        if (episode+1) % 10 == 0:
            avg_reward = np.mean(total_rewards_list[-10:])
            print(f"[Episode {episode+1}] Avg Reward(last10)={avg_reward:.2f}, Epsilon={epsilon:.3f}")

            if save_render:
                # Save files
                ep_folder = os.path.join(render_path, f"episode_{episode+1}")
                os.makedirs(ep_folder, exist_ok=True)
                for i, (img, txt, state_text) in enumerate(zip(episode_frames, episode_info_texts, episode_state_texts)):
                    imageio.imwrite(os.path.join(ep_folder, f"step_{i:03d}.png"), img)
                    with open(os.path.join(ep_folder, f"step_{i:03d}.info.txt"), "w", encoding="utf-8") as f:
                        f.write(txt)
                    with open(os.path.join(ep_folder, f"step_{i:03d}.state_texts.txt"), "w", encoding="utf-8") as f:
                        f.write(state_text)

    env.close()
    return dqn, total_rewards_list

# Generating a GIF
You can compile saved PNG frames into a GIF for a chosen episode.
Below are the functions for overlaying text onto images and then creating a GIF:

## Overlay Text on Images

In [ ]:
def overlay_text_on_image(image_path, info_path, font_path=None):
   """
   image_path: Path to PNG file
   info_path: Path to corresponding .info.txt file
   font_path: Path to TTF font (optional)
   return: Pillow Image object (image with overlaid text)
   """
   img = Image.open(image_path).convert("RGB")
   draw = ImageDraw.Draw(img)

   # Set font (use default if none provided)
   if font_path is not None:
       font = ImageFont.truetype(font_path, size=18)
   else:
       font = ImageFont.load_default()

   text = ""
   if os.path.exists(info_path):
       with open(info_path, "r", encoding="utf-8") as f:
           text = f.read()

   # Display multi-line text with line breaks
   lines = text.strip().split("\n")

   # Set position
   x, y = 10, 10
   padding = 5  # Padding for black box
   max_width = max([draw.textlength(line, font=font) for line in lines]) + 2 * padding
   box_height = len(lines) * 20 + 2 * padding  # Adjust box size based on number of lines

   # Semi-transparent black box (improves readability)
   draw.rectangle([(x - padding, y - padding), (x + max_width, y + box_height)], fill=(0, 0, 0, 150))

   # Add text (yellow)
   for line in lines:
       draw.text((x, y), line, fill=(255, 255, 0), font=font)
       y += 20  # Line spacing

   return img

## Create a GIF

In [ ]:
def make_gif_with_overlay(ep_folder, output_gif="cartpole_overlay.gif", fps=30, font_path=None):
   """
   ep_folder: Path to 'episode_N' directory (containing PNG & info.txt files)
   output_gif: Path for output GIF file to be created
   font_path: Path to TTF font (uses default font if not provided)
   """
   # Sort PNG files (step_000.png, step_001.png ...)
   file_list = sorted([f for f in os.listdir(ep_folder) if f.endswith(".png")])
   frames = []

   for file_name in file_list:
       image_path = os.path.join(ep_folder, file_name)
       info_path = image_path.replace(".png", ".info.txt")
       # Create image with text overlay
       annotated_img = overlay_text_on_image(image_path, info_path, font_path=font_path)
       frames.append(np.array(annotated_img))  # Convert to numpy array

   if len(frames) > 0:
       imageio.mimsave(output_gif, frames, fps=fps)
       print(f"GIF saved: {output_gif}")
   else:
       print(f"No PNG files found in {ep_folder}.")

# Training

In [ ]:
dqn_model, rewards = train_dqn(
    env_name="CartPole-v1",
    num_episodes=320,
    save_render=True,
    render_path="./cartpole_text_frames"
)

In [ ]:
ep_folder = "./cartpole_text_frames/episode_320"
make_gif_with_overlay(ep_folder, output_gif="cartpole_ep20.gif", fps=30)

In [ ]:
from IPython.display import Image as IPyImage

In [ ]:
IPyImage(filename="cartpole_ep320.gif")

# Conclusion
In this text-based DQN example, we showcased how to:

1. Wrap the CartPole environment to convert numeric states into text.
2. Embed those text states using DistilBERT.
3. Train a DQN to handle textual observations.
4. Visualize the environment’s rendered frames alongside textual details.

This highlights how reinforcement learning can be extended to non-visual, textual state representations.

You can further experiment with:
- Larger or custom language models.
- More sophisticated text representations.
- Combining text with numeric or image data in a multimodal RL framework.